In [2]:
import requests
import json
import pandas as pd
import time
import pprint as pp
import re

### Steps to retrieve ClickUp Data

If you want to check how the data is extracted from click up, uncomment all callable codes from the comments & run them until the next markdown

In [3]:
# config.py
CLICKUP_TOKEN = 'pk_80488909_E7HYSNIATQDY0IJF3894MGZXKGDNQAJA'
HEADERS = {"Authorization": CLICKUP_TOKEN}

# IDs - keep them in one place for easy changes
TEAM_ID = "9012223014"
BESA_SPACE_ID = "90121722741"
OPERATIONS_FOLDER_ID = "90123018368"
BESA_CLIENTS_LIST_ID = "901205091637"

USER > WORKSPACE > FOLDER > LISTS > TASKS 

In [4]:
def clickup_get(endpoint, headers, params=None, delay=0.3):
    url = f"https://api.clickup.com/api/v2/{endpoint}"
    res = requests.get(url, headers=headers, params=params)
    if res.status_code != 200:
        raise Exception(f"ClickUp API error: {res.status_code} - {res.text}")
    time.sleep(delay)  # avoid rate limit
    return res.json()

In [5]:
# GET USER ID
user = clickup_get(f"user", HEADERS)
# user

In [6]:
# GET MAPOGOS TEAM ID
team = clickup_get(f"team", HEADERS)
# team

# -> Mapogos Capital has TEAM_ID: 9012223014

In [7]:
# GET BESA CONSTRUCTION WORKSPACE ID
besa_space = clickup_get(f"team/{TEAM_ID}/space", HEADERS)
# besa_space

# -> Besa Construction Workspace has BESA_SPACE_ID: 90121722741

In [8]:
# GET BESA CONSTRUCTION OPERATIONS FOLDER ID
besa_operations = clickup_get(f"space/{BESA_SPACE_ID}/folder", HEADERS)
# besa_operations

# -> Besa has OPERATIONS_FOLDER_ID: 90123018368

In [9]:
CLICKUP_TOKEN = 'pk_80488909_E7HYSNIATQDY0IJF3894MGZXKGDNQAJA'
HEADERS = {"Authorization": CLICKUP_TOKEN}

In [10]:
res = requests.get("https://api.clickup.com/api/v2/list/901205091637/task", headers=HEADERS)
res.json()

{'tasks': [{'id': '869c5pbhm',
   'custom_id': None,
   'custom_item_id': 0,
   'name': 'Simon Richardson #248',
   'text_content': '',
   'description': '',
   'status': {'status': 'completed project not paid',
    'id': 'sc901205091637_wB22ZYIo',
    'color': '#d33d44',
    'type': 'done',
    'orderindex': 4},
   'orderindex': '120302423.00009700000000000000000000000000',
   'date_created': '1771281039387',
   'date_updated': '1771614958479',
   'date_closed': None,
   'date_done': '1771595576455',
   'archived': False,
   'creator': {'id': 152482999,
    'username': 'Dardan Krasniqi ',
    'color': None,
    'email': 'contact@besaconstruction.co.uk',
    'profilePicture': None},
   'assignees': [],
   'group_assignees': [],
   'watchers': [{'id': 152482999,
     'username': 'Dardan Krasniqi ',
     'color': None,
     'initials': 'DK',
     'email': 'contact@besaconstruction.co.uk',
     'profilePicture': None},
    {'id': 80488867,
     'username': 'Nicole',
     'color': '',
    

In [27]:
# GET BESA CLIENTS LIST FROM OPERATIONS FOLDER
besa_client_list = clickup_get(f"folder/{OPERATIONS_FOLDER_ID}/list", HEADERS)

# -> Besa has BESA_CLIENTS_LIST_ID : 901205091637

#### RETRIEVE ALL BESA PROJECTS

In [26]:
# GET ALL PROJECTS/TASKS FROM BESA CLIENTS FOLDER
tasks = clickup_get(f"list/{BESA_CLIENTS_LIST_ID}/task", HEADERS)["tasks"]


In [13]:
for task in tasks:
    if task['id'] == '869a4xzde':
        print(task['text_content'])

Date
Dardan Labor
Musa Labor
REMZI 
Dori Labor 
Daily Progress
 16.08.25
 in
 off

 in
set up
 18.08.25
 in
 off

 in
protection
 19.08.25
 in
 in

 in
-
20.08.25
 in
 in

 in
-
21.08.25
in
in

in
-
22.08.25
in
in

in
painting
23.08.25
in
in

in
painting
25.08.25
in
in

in
painting
26.08.25
in
in

in
painting
27.08.25
off
off

off
-
28.08.25
in
in

in
painting
29.08.25
in
in

in
painting
30.08.25
in
in

in
painting
31.08.25
in
in

in
painting
05.09.25
IN
IN

IN
PAINTING 
06.09.25
IN
IN

IN
PAINTING 
07.09.25
in
in

in
painting
08.09.25
IN
IN

IN
PAINTING
09.09.25
IN
IN
IN
IN
PAINTING
10.09.25
IN
IN
IN
IN
PAINTING
11.09.25
in
in
in
in
painting
12.09.25


in


15.09.25
IN
IN
IN
HOLLIDAY
PAINTING
16.09.25
IN
IN



17.09.25
in


### NOW ADDING MODULARITY

In [14]:
# extract.py
from api import clickup_get
from config import HEADERS
import re
 
# get_paid_invoice_tasks prints all the paid invoice project ids line by line
def get_all_tasks(list_id):
    # to return just paid invoices
    tasks = clickup_get(f"list/{list_id}/task", HEADERS)["tasks"]
    return [t for t in tasks]

def get_task_details(task_ids):
    details = []
    for task_id in task_ids:
        details.append(clickup_get(f"task/{task_id}", HEADERS))
    return details

def get_subtasks(list_id):
    subtasks = []
    tasks_with_subtasks = clickup_get(
        f"list/{list_id}/task",
        HEADERS,
        params={"subtasks": "true"}
    )["tasks"]
    for task in tasks_with_subtasks:
        if task.get("parent"):  # this task is a subtask
            subtasks.append(task)
    return subtasks

def get_task_texts(tasks):
    """
    Retrieves the 'text_content' (or fallback 'description')
    for ALL tasks in the list.
    Returns a dict {task_id: text_content}.
    """
    results = {}
    for task in tasks:
        text = task.get('text_content') or task.get('description') or ""
        results[task['id']] = text
    return results

import pandas as pd

def labor_text_to_table(labor_text, task_id):
    """
    Converts labor activity text into a structured DataFrame.
    Assumes format:
        Date
        Dardan Labor
        Musa Labor
        Dori Labor
        Daily Progress
        <rows in multiples of 5>
    """
    # Split lines and remove empty lines
    lines = [line.strip() for line in labor_text.splitlines() if line.strip()]

    if len(lines) < 6:
        return pd.DataFrame()  # not enough data to form a table

    # Header
    header = lines[:5]
    rows = lines[5:]

    # Group rows by 5
    data = [rows[i:i+5] for i in range(0, len(rows), 5)]

    # Build DataFrame
    df = pd.DataFrame(data, columns=header)

    # Clean strings
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    # Attach project_id
    df["project_id"] = task_id

    return df

In [15]:
all_tasks = get_all_tasks(BESA_CLIENTS_LIST_ID)
paid_tasks = [t for t in all_tasks if t["status"]["status"].lower() == "paid invoice"]
ongoing_tasks = [t for t in all_tasks if t["status"]["status"].lower() == "ongoing"]

In [ ]:
labor_text = get_task_texts(ongoing_tasks)

{'869b1hnkk': 'Date\nDardan Labor\nMusa Labor\nDori Labor \nDaily Progress\n\xa019.01.26\n\xa0in\n\xa0in\n\xa0of\n\xa0sti=ripin wal\n\xa020.01.25\n\xa0in\n\xa0in\n\xa0in\n\xa0striping wall primer\n\xa021.01.25\n\xa0in\n\xa0in\n\xa0in\n\xa0door and electrick\n\xa022.01.25\n\xa0in\n\xa0in\n\xa0in\n\xa0pointing and paint \n23.01.25\nin\nin\nin\npaint wall\n24.01.25\n\nin\nin\npaint wall\n28.01.25\nin\nin\noff\nlead & paint wall\n29.01.25\nin\nin\nin\ncctv.guttering .front wall filler \n30.01.25\nin 0.5\nin 0.5\nin 0.5\nwindow kitchen \n02.02.25\nin\nin\nin\nbath & bed window \n09.02.26\n0.5\n0.5\n0.5',
 '869ar19y7': ''}

paid_tasks → "summary" tasks from the list query.


detailed_tasks → "full task details" fetched individually.

All BESA PROJECT IDs

In [17]:
# Retrieve all the task_details, all of them will take a very long time
detailed_tasks = get_task_details([t["id"] for t in paid_tasks])
detailed_tasks

[{'id': '869c1tu79',
  'custom_id': None,
  'custom_item_id': 0,
  'name': 'PARSONS NOSE  (Putney) #246',
  'text_content': '',
  'description': '',
  'status': {'id': 'sc901205091637_xkyptcu1',
   'status': 'paid invoice',
   'color': '#008844',
   'orderindex': 5,
   'type': 'done'},
  'orderindex': '120302364.00016050000000000000000000000000',
  'date_created': '1770311423708',
  'date_updated': '1770654140797',
  'date_closed': None,
  'date_done': '1770311423708',
  'archived': False,
  'creator': {'id': 152482999,
   'username': 'Dardan Krasniqi ',
   'color': None,
   'email': 'contact@besaconstruction.co.uk',
   'profilePicture': None},
  'assignees': [],
  'group_assignees': [],
  'watchers': [{'id': 80488867,
    'username': 'Nicole',
    'color': '',
    'initials': 'N',
    'email': 'msnicolebaro@gmail.com',
    'profilePicture': 'https://attachments.clickup.com/profilePictures/80488867_lun.jpg'},
   {'id': 152482999,
    'username': 'Dardan Krasniqi ',
    'color': None,
 

## Helpers

In [18]:
import re
import pandas as pd
import datetime
from extract import get_task_texts, labor_text_to_table
# ----------------------
# Helpers
# ----------------------
def get_custom_field_value(custom_fields, field_name):
    for field in custom_fields:
        if field.get("name") == field_name:
            return field.get("value")
    return None

def get_custom_field_option_name(custom_fields, field_name):
    for field in custom_fields:
        if field.get("name") == field_name and "value" in field:
            value = field["value"]
            for option in field.get("type_config", {}).get("options", []):
                if option.get("orderindex") == value:
                    return option["name"]
    return None

def safe_timestamp_to_date(ts):
    if ts:
        return datetime.datetime.fromtimestamp(int(ts) / 1000).strftime('%Y-%m-%d')
    return None


## Projects

In [19]:
# ----------------------
# Projects Table
# ----------------------
def build_projects_table(tasks):
    data = []
    for task in tasks:
        cf = task.get("custom_fields", [])
        row = {
            "project_id": task.get("id"),
            "client_name": get_custom_field_value(cf, "Client Name"),
            "sales_channel": get_custom_field_option_name(cf, "How did you hear about us?"),
            "project_name": task.get("name"),
            "client_address": get_custom_field_value(cf, "Project Address"),
            "client_email_address": get_custom_field_value(cf, "Email Address"),
            "client_phone_number": get_custom_field_value(cf, "Phone Number"),
            "project_status": task.get("status", {}).get("status"),
            "project_start_date": safe_timestamp_to_date(task.get("start_date")),
            "project_end_date": safe_timestamp_to_date(task.get("due_date")),
            "project_value": get_custom_field_value(cf, "Total Price (excl. VAT%)"),
            "dardan_days_worked": get_custom_field_value(cf, "01 Dardan (days worked)"),
            "musa_days_worked": get_custom_field_value(cf, "02 Musa (days worked)"),
            "dori_days_worked": get_custom_field_value(cf, "03 Dori (days worked)"),
            "remzi_days_worked": get_custom_field_value(cf, "04 Remzi (days worked)"),
        }
        data.append(row)

    projects_df = pd.DataFrame(data)
    return projects_df

raw_project_df = build_projects_table(detailed_tasks)


## Clean Projects

In [20]:
# ----------------------
# Cleaning Projects Table
# ----------------------

def clean_project_addresses(df):
    # Flexible UK postcode regex
    postcode_regex = r'([A-Z]{1,2}\d{1,2}[A-Z]?\s*\d[A-Z]{2})'
    
    def extract_parts(address):
        if not isinstance(address, str):
            return None, None, None, None, None
        
        # Normalize spaces & uppercase
        addr = re.sub(r'\s+', ' ', address.strip()).upper()
        
        # Find postcode
        m = re.search(postcode_regex, addr, re.IGNORECASE)
        if not m:
            return addr, None, None, None, None
        
        postcode = m.group(1).strip()
        
        # Split outward & inward
        m2 = re.match(r'([A-Z]{1,2}\d{1,2}[A-Z]?)\s*(\d[A-Z]{2})', postcode)
        outward = m2.group(1).upper() if m2 else None
        inward = m2.group(2).upper() if m2 else None
        
        # Break address into parts
        before_postcode = addr[:m.start()].strip(" ,")
        after_postcode = addr[m.end():].strip(" ,")

        city = None
        street = before_postcode
        
        # Case 1: city is before postcode (look for last comma section)
        if ',' in before_postcode:
            tokens = [t.strip() for t in before_postcode.split(',')]
            street = ', '.join(tokens[:-1])
            city = tokens[-1]
        
        # Case 2: city is after postcode
        if after_postcode:
            city = after_postcode
        
        return street, postcode, outward, inward, city, addr
    
    
    parts = df['client_address'].apply(lambda x: pd.Series(extract_parts(x)))
    parts.columns = ['client_street_name', 
                     'client_post_code', 
                     'client_outward_code', 
                     'client_inward_code', 
                     'client_city',
                     'client_address']
    
    return pd.concat([df.drop(columns=[
        "client_street_name", 
        "client_post_code", 
        "client_outward_code", 
        "client_inward_code", 
        "client_city", 
        "client_address"
    ], errors="ignore"), parts], axis=1)

cln_project_df = clean_project_addresses(raw_project_df)


## Materials

In [21]:
# ----------------------
# Materials Table
# ----------------------
def build_materials_table(tasks):
    material_data = []
    for task in tasks:
        project_id = task.get("id")
        project_name = task.get("name")
        for checklist in task.get("checklists", []):
            for item in checklist.get("items", []):
                raw_name = item.get("name", "").strip()
                match_full = re.match(r"^(.*?)[\s\-]+(\d+(?:\.\d+)?)[\s\-]+(\d+)$", raw_name)
                match_cost_only = re.match(r"^(.*?)[\s\-]+(\d+(?:\.\d+)?)$", raw_name)

                if match_full:
                    expense_name, expense_cost, quantity = match_full.group(1).strip(), float(match_full.group(2)), int(match_full.group(3))
                elif match_cost_only:
                    expense_name, expense_cost, quantity = match_cost_only.group(1).strip(), float(match_cost_only.group(2)), 1
                else:
                    expense_name, expense_cost, quantity = raw_name, None, None

                material_data.append({
                    "project_id": project_id,
                    "project_name": project_name,
                    "expense_name": expense_name,
                    "expense_cost": expense_cost,
                    "quantity": quantity
                })

    material_df = pd.DataFrame(material_data)
    return material_df

material_df = build_materials_table(detailed_tasks)
material_df

,project_id,project_name,expense_name,expense_cost,quantity
0,869c1tu79,PARSONS NOSE (Putney) #246,hps boiler,312.72,1.0
1,869c1tu79,PARSONS NOSE (Putney) #246,travis perkings,12.92,1.0
2,869bzfqnd,"SIGNE ,RUBER ROFF",raven ruber,480.95,1.0
3,869bzfqnd,"SIGNE ,RUBER ROFF",lords ply,31.44,1.0
4,869bzfqnd,"SIGNE ,RUBER ROFF",zyber proteck,15.39,1.0
...,...,...,...,...,...
278,8698kr34y,PARSONS NOSE FULHAM (FRAM BOX) 5 HEATHMANS ROA...,metal frame steel,84.19,1.0
279,8698kr34y,PARSONS NOSE FULHAM (FRAM BOX) 5 HEATHMANS ROA...,tools,9.88,1.0
280,8698kqvp0,SIOBHAN (LIGHT WELLS) 33 CLONCURRY STREET #174,materials,70.50,1.0
281,8698kqvp0,SIOBHAN (LIGHT WELLS) 33 CLONCURRY STREET #174,Floor Protection Materials,35.00,1.0


## Services

In [22]:
def build_services_table(tasks):
    # ✅ filter projects with paid invoice status
    paid_projects = [
        t for t in tasks 
        if t.get("status", {}).get("status", "").lower() == "paid invoice"
    ]

    # map only paid projects
    task_id_to_name = {task["id"]: task["name"] for task in paid_projects}
    paid_project_ids = set(task_id_to_name.keys())

    subtasks = get_subtasks(BESA_CLIENTS_LIST_ID)
    subtasks_data = []

    for subtask in subtasks:
        parent_id = subtask.get("parent")

        # ✅ only keep subtasks whose parent is a paid project
        if parent_id in paid_project_ids:
            cf = subtask.get("custom_fields", [])
            project_name = task_id_to_name.get(parent_id)

            subtasks_data.append({
                "project_id": parent_id,
                "project_name": project_name,
                "project_status": "paid invoice",  # since we already filtered
                "subtask_id": subtask["id"],
                "subtask_name": subtask["name"],
                "service_price": get_custom_field_value(cf, "Total Price (excl. VAT%)"),
                "service_description": get_custom_field_value(cf, "Service Description"),
            })

    subtasks_df = pd.DataFrame(subtasks_data)
    return subtasks_df

build_services_table(detailed_tasks)

,project_id,project_name,project_status,subtask_id,subtask_name,service_price,service_description
0,869b7au2n,SIGNE. 69 RUSHMOLE ROAD #229,paid invoice,869brtmqz,top flore roof,530,None
1,869b7au2n,SIGNE. 69 RUSHMOLE ROAD #229,paid invoice,869brtmp4,deking,None,None
2,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bgq8tf,Living room door Relocated,280,None
3,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bgq3au,Storage unit doors 2 number,380,None
4,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bgq1mw,Front door paint,480,None
5,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bdhpzt,paint laundry room,750,None
6,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bdhpx2,garage door from out site paint,680,None
7,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bdhpuu,kichen handels,320,None
8,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bdhptv,pipe draning block,365,None
9,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bdhprh,loft bords,530,None


In [23]:
def build_services_table(tasks, status="paid invoice"):
    # ✅ filter projects by given status
    filtered_projects = [
        t for t in tasks 
        if t.get("status", {}).get("status", "").lower() == status.lower()
    ]

    # map project id → (name, status)
    task_id_to_meta = {
        task["id"]: {
            "name": task["name"],
            "status": task.get("status", {}).get("status", "")
        }
        for task in filtered_projects
    }
    project_ids = set(task_id_to_meta.keys())

    subtasks = get_subtasks(BESA_CLIENTS_LIST_ID)
    subtasks_data = []

    for subtask in subtasks:
        parent_id = subtask.get("parent")

        if parent_id in project_ids:
            cf = subtask.get("custom_fields", [])
            parent_meta = task_id_to_meta[parent_id]

            subtasks_data.append({
                "project_id": parent_id,
                "project_name": parent_meta["name"],
                "project_status": parent_meta["status"],     # 👈 project status
                "subtask_id": subtask["id"],
                "subtask_name": subtask["name"],
                "subtask_status": subtask.get("status", {}).get("status", ""),  # 👈 subtask status
                "service_price": get_custom_field_value(cf, "Total Price (excl. VAT%)"),
                "service_description": get_custom_field_value(cf, "Service Description"),
            })

    subtasks_df = pd.DataFrame(subtasks_data)
    return subtasks_df

build_services_table(detailed_tasks)


,project_id,project_name,project_status,subtask_id,subtask_name,subtask_status,service_price,service_description
0,869b7au2n,SIGNE. 69 RUSHMOLE ROAD #229,paid invoice,869brtmqz,top flore roof,warm lead,530,None
1,869b7au2n,SIGNE. 69 RUSHMOLE ROAD #229,paid invoice,869brtmp4,deking,warm lead,None,None
2,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bgq8tf,Living room door Relocated,warm lead,280,None
3,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bgq3au,Storage unit doors 2 number,warm lead,380,None
4,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bgq1mw,Front door paint,warm lead,480,None
5,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bdhpzt,paint laundry room,warm lead,750,None
6,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bdhpx2,garage door from out site paint,warm lead,680,None
7,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bdhpuu,kichen handels,warm lead,320,None
8,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bdhptv,pipe draning block,warm lead,365,None
9,869a404c4,VICTORIA NGUYEN 60 EATONS MEWS NORTH #203,paid invoice,869bdhprh,loft bords,warm lead,530,None


In [24]:
# ----------------------
# Services Table
# ----------------------
def build_services_table(tasks):
    subtasks = get_subtasks(BESA_CLIENTS_LIST_ID)
    task_id_to_name = {task["id"]: task["name"] for task in detailed_tasks}
    subtasks_data = []
    for subtask in subtasks:
            if subtask.get("parent"):
                    cf = subtask.get("custom_fields",[])
                    parent_id = subtask['parent']
                    project_name = task_id_to_name.get(parent_id)
            subtasks_data.append({
                    "project_id": parent_id,
                    "project_name": project_name,
                    "project_status": task["status"],
                    "subtask_id": subtask["id"],
                    "subtask_name": subtask['name'],
                    "service_price": get_custom_field_value(cf,'Total Price (excl. VAT%)'),
                    "service_description": get_custom_field_value(cf,'Service Description')
            })
            subtasks_df = pd.DataFrame(subtasks_data)
    subtasks_df = pd.DataFrame(subtasks_data)
    return subtasks_df

build_services_table(detailed_tasks)

,project_id,project_name,project_status,subtask_id,subtask_name,service_price,service_description
0,869c593gv,None,"{'status': 'no response/refusal', 'id': 'sc901...",869c596p1,Send a kind introductory email,None,None
1,869c3y2xp,None,"{'status': 'no response/refusal', 'id': 'sc901...",869c3y3ty,Jet wash the deck,None,None
2,869c1mc94,None,"{'status': 'no response/refusal', 'id': 'sc901...",869c3y3kz,Spiral staircase to roof decking:,None,None
3,869c1mc94,None,"{'status': 'no response/refusal', 'id': 'sc901...",869c3y3f8,Paint in the front house,None,None
4,869c1mc94,None,"{'status': 'no response/refusal', 'id': 'sc901...",869c3y2xp,Opening wall between the kitchen and the dini...,None,None
...,...,...,...,...,...,...,...
57,869auttbk,TAUSIF AWAN (304 MALDEN ROAD KT3 6AT) #219,"{'status': 'no response/refusal', 'id': 'sc901...",869avrv07,rendering wall outsite,None,None
58,869atzef7,None,"{'status': 'no response/refusal', 'id': 'sc901...",869atzegj,Service 4 - Biding,80,None
59,869atzef7,None,"{'status': 'no response/refusal', 'id': 'sc901...",869atzegh,"Service 5 - Supply, build and paint custom sho...",320,None
60,869atzef7,None,"{'status': 'no response/refusal', 'id': 'sc901...",869atzege,Service 3 - Tiles Extension,120,None


## Labor

In [ ]:
# ----------------------
# Labor Table
# ----------------------
def build_labor_table(tasks, existing_df=None):
    """
    Loops through all tasks and combines their labor logs into one DataFrame.
    Keeps existing labor data (if provided) and appends new tasks.
    
    Parameters:
        tasks (list): List of task dictionaries
        existing_df (pd.DataFrame, optional): Existing labor table to keep
    
    Returns:
        pd.DataFrame: Updated labor table sorted by Date
    """
    task_texts = get_task_texts(tasks) 
    all_labor = []

    # Keep existing data if provided
    if existing_df is not None:
        all_labor.append(existing_df)
        existing_task_ids = set(existing_df['project_id'].unique())
    else:
        existing_task_ids = set()

    for task_id, text in task_texts.items():
        # Only process tasks that are not already in existing_df
        if task_id not in existing_task_ids and text.strip():  
            df = labor_text_to_table(text, task_id) 
            if not df.empty: 
                all_labor.append(df)

    if all_labor: 
        labor_df = pd.concat(all_labor, ignore_index=True)

        # Convert 'Date' to datetime and sort chronologically
        labor_df['Date'] = pd.to_datetime(labor_df['Date'], format='%d.%m.%y')
        labor_df = labor_df.sort_values('Date').reset_index(drop=True)

        return labor_df

    return existing_df if existing_df is not None else pd.DataFrame()

labor_df = build_labor_table(ongoing_tasks)
labor_df

^^ This didn't work

## CLEANING - Normalise tables into 3rd Normal Form

#### Client Dimension 

In [28]:
# --- Client Dimension ---
client_dim = cln_project_df[["client_name",
                        "sales_channel",
                        "client_email_address",
                        "client_phone_number",
                        "client_street_name",
                        "client_post_code",
                        "client_outward_code",
                        "client_inward_code",
                        "client_city"]]
client_dim = client_dim.drop_duplicates(
    subset=["client_name", "client_email_address", "client_post_code"],
    keep="first"
).reset_index(drop=True)
client_dim['client_numeric_id'] = client_dim.index + 1
client_dim['client_id'] = client_dim['client_numeric_id'].apply(lambda x: f"C{x:03d}")  # e.g., C001
client_dim.drop(['client_numeric_id'],axis=1,inplace=True)
client_dim.dropna(axis=0, how="all",inplace=True)

In [29]:
client_dim

,client_name,sales_channel,client_email_address,client_phone_number,client_street_name,client_post_code,client_outward_code,client_inward_code,client_city,client_id
0,None,None,None,None,None,None,None,None,None,C001
1,Axel Mathysen Gerst,None,Axelamg@hotmail.com,None,7A SCHUBERT ROAD,SW15 2QT,SW15,2QT,LONDON,C002
2,ELAINE KLANDER,From a Friend,Elaine@klander.com,+44 7957 165393,36 SCHOOL ROAD EAST MOLESEY,KT8 0DN,KT8,0DN,None,C003
3,TONI Hindhaugh,Existing Client,catherine@parsonsnose.co.uk,None,753 FULHAM ROAD,SW6 5UU,SW6,5UU,LONDON,C004
4,None,None,tpk@colesbourne.com,+44 7801 430020,103 NEW KINS ROAD,None,None,None,None,C005
5,VICTORIA NGUYEN,None,victoriakieuanhnguyen@gmail.com,+44 7909 785501,60 EATONS MEWS NORTH,SW1X 8LL,SW1X,8LL,None,C006
6,MATTHEW ROYLE,Existing Client,Oxberryfulham@gmail.com,None,11 CHEPSTOW CLOSE,SW15 2HG,SW15,2HG,PUTNEY,C007
7,IMOGEN,None,imogen.watson@btopenworld.com,+44 7766 857638,10A LOUVAINE ROAD,SW11 2AQ,SW11,2AQ,None,C008
8,KATY,Existing Client,info@katyellisinteriordesign.com,+447896946015,3 RUSHHILL MEWS,SW11 5NB,SW11,5NB,None,C009
9,ANNA ( JAN ),Existing Client,Acpare@gmail.com,+44 7802 980660,45 CALDERVALE ROAD,SW8 9LY,SW8,9LY,LONDON,C010


Connect to BigQuery 

In [30]:
from google.cloud import bigquery

client = bigquery.Client(project="besa-construction-database")
sql = "SELECT * FROM `clean_dataset.ClientDim` LIMIT 10"
df = client.query(sql).to_dataframe()
print(df.head())


/Users/madalinasamoila/besa_pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  string_field_0  string_field_1   string_field_2          string_field_3  \
0           C025            None             None                    None   
1           C009  tony hindhaugh  Existing Client  tony@parsonsnose.co.uk   
2           C032  James Sinclair  Existing Client        jimsin12@aol.com   
3           C010       Tom King   Existing Client     tpk@colesbourne.com   
4           C017         Shabhan    Word Of Mouth    shaban@vitalbs.co.uk   

    string_field_4                                     string_field_5  \
0             None                                               None   
1  +44 7717 750885                                               None   
2             None  14 ROYSTON COURT HOOK RISE, NORTH SURBITON, KT...   
3  +44 7801 430020                 22 ANSTIS STREET, PLYMOUTH, PL15JT   
4             None                36 KERSLEY STREET, LONDON, SW11 4PT   

               string_field_6 string_field_7 string_field_8 string_field_9  \
0                   

In [ ]:
import os

cur_path = os.getcwd()
file ='ProjectDim.csv'
file_path = os.path.join(cur_path,"data/clean",file)
print(file_path)

/Users/madalinasamoila/besa_pipeline/data/clean/ProjectDim.csv


### LoadJobConfig

https://cloud.google.com/bigquery/docs/reference/rest/v2/Job#JobConfigurationLoad.FIELDS.write_disposition

In [31]:
# modules
from google.cloud import bigquery
import os

# Instantiate our client 
# Define where our data should be loaded to (in this case google cloud -> besa-construction -> clean_dataset -> ProjectDim)
client = bigquery.Client(project="besa-construction-database")
target_table = "besa-construction-database.clean_dataset.ProjectDim"

# Defining the Job Config for how the data should be inserted
job_config = bigquery.LoadJobConfig(
    skip_leading_rows=1,
    source_format=bigquery.SourceFormat.CSV,
    autodetect= True,
    write_disposition='WRITE_TRUNCATE'
)

# Set the path to the file
cur_path = os.getcwd()
file ='ProjectDim.csv'
file_path = os.path.join(cur_path,"data/clean",file)
print(file_path)

# Open the file, add parameters
with open(file_path,'rb') as source_file:
    load_job = client.load_table_from_file(
        source_file,
        target_table,
        job_config=job_config
    )

load_job.result()

destination_table = client.get_table(target_table)
print(f"You have {destination_table.num_rows} rows in your table")


/Users/madalinasamoila/besa_pipeline/data/clean/ProjectDim.csv
You have 62 rows in your table


### Load all the rest of the CSV files

In [32]:
# modules
from google.cloud import bigquery
import os

# Instantiate our client 
# Define where our data should be loaded to (in this case google cloud -> besa-construction -> clean_dataset -> ProjectDim)
client = bigquery.Client(project="besa-construction-database")

# List of CSV files and thier corresponding target tables
csv_table_mapping = {
    "ClientDim.csv": "besa-construction-database.clean_dataset.ClientDim",
    "ProjectDim.csv": "besa-construction-database.clean_dataset.ProjectDim",
    "ExpenseFact.csv": "besa-construction-database.clean_dataset.ExpenseFact",
    "ProjectFact.csv": "besa-construction-database.clean_dataset.ProjectFact"
}

# Defining the Job Config for how the data should be inserted
job_config = bigquery.LoadJobConfig(
    skip_leading_rows=1,
    source_format=bigquery.SourceFormat.CSV,
    autodetect= True,
    write_disposition='WRITE_TRUNCATE'
)

# Loop through each file and load it into BigQuery


cur_path = os.getcwd()
file ='ProjectDim.csv'
file_path = os.path.join(cur_path,"data/clean",file)
print(file_path)

# Open the file, add parameters
with open(file_path,'rb') as source_file:
    load_job = client.load_table_from_file(
        source_file,
        target_table,
        job_config=job_config
    )

load_job.result()

destination_table = client.get_table(target_table)
print(f"You have {destination_table.num_rows} rows in your table")


/Users/madalinasamoila/besa_pipeline/data/clean/ProjectDim.csv
You have 62 rows in your table


In [33]:
for file_name, target_table in csv_table_mapping.items():
    file_path = os.path.join(cur_path,"data/clean",file_name)
    print(f"Loading {file_path} into {target_table}...")

    with open(file_path,'rb') as source_file:
        load_job = client.load_table_from_file(
        source_file,
        target_table,
        job_config=job_config
    )

    load_job.result()

    destination_table = client.get_table(target_table)
    print(f"You have {destination_table.num_rows} rows in your {target_table} table")


Loading /Users/madalinasamoila/besa_pipeline/data/clean/ClientDim.csv into besa-construction-database.clean_dataset.ClientDim...
You have 32 rows in your besa-construction-database.clean_dataset.ClientDim table
Loading /Users/madalinasamoila/besa_pipeline/data/clean/ProjectDim.csv into besa-construction-database.clean_dataset.ProjectDim...
You have 62 rows in your besa-construction-database.clean_dataset.ProjectDim table
Loading /Users/madalinasamoila/besa_pipeline/data/clean/ExpenseFact.csv into besa-construction-database.clean_dataset.ExpenseFact...
You have 263 rows in your besa-construction-database.clean_dataset.ExpenseFact table
Loading /Users/madalinasamoila/besa_pipeline/data/clean/ProjectFact.csv into besa-construction-database.clean_dataset.ProjectFact...
You have 60 rows in your besa-construction-database.clean_dataset.ProjectFact table


In [34]:
destination_table = client.get_table(target_table)
destination_table

Table(TableReference(DatasetReference('besa-construction-database', 'clean_dataset'), 'ProjectFact'))

In [35]:
with open(file_path) as source_file:
    print(source_file.read())

project_id,project_name,project_value,dardan_days_worked,musa_days_worked,dori_days_worked,remzi_days_worked,total_material_cost
869a9uhvd,EXEL (7A SCHUBERT ROAD SW15 2QT) #210,3040,2,2,2,,192.21000000000004
869a9a7rn,FRANCESCA (46 HORNTON STREET ),180,0.2,0.2,0.2,,0.0
869a61xwk,MARTIN BROWNE (46 WATERFORD ROAD) #205,1420,0.5,0.5,0.5,,128.38
869a588v6,MARTIN (46 WATERFORD ROAD) #204,270,0.2,,0.2,,80.0
869a12e5z,DAVID (SUPPLY & INSTALL TIMBER STUD) 15 DEANS ROAD #143,4250,1,1,1,0,
8699zaq36,LAURA (40 STOKENCHURCH STREET) #202,2400,2,2,2,,202.48
8699w2nyd,PARSONS NOSE BELGRAVIA (FIX SINK & PIPE) 81 EBURY STREET  #200,350,1,0,0,0,
8699nmjq2,MARTIN 46 WATERFORD ROAD #198,1000,1,0,0,0,82.39999999999999
8699ftxm9,ANNA (PAINTING) 45 CALDERVALE ROAD #189,590,0.5,0,0,0.5,85.0
8699cky12,KATE 35 CLONCURRY STREET #300,2600,2,2,2,,378.4
86999z3pc,PARSONS NOSE SCOTLAND,11000,6,13,13,0,0.0
86999ywg8,TOM KING 103 NEW KINGS ROAD #194,5575,4,0,0,8,3111.93
86999yv2u,KENDALL (ONE DAY) 12 DONARELLE STREET 

#### Project Dimension 

In [36]:
# --- Project Dimension ---
project_dim = cln_project_df[[
        "project_id", 
        "project_name", 
        "project_status",
        "project_start_date", 
        "project_end_date", 
        "client_name",
        "client_post_code"
]].merge(client_dim[["client_id","client_name","client_post_code"]], on=["client_name", "client_post_code"], how="left")

project_dim = project_dim.drop(columns=["client_name","client_post_code"])
project_dim


,project_id,project_name,project_status,project_start_date,project_end_date,client_id
0,869c1tu79,PARSONS NOSE (Putney) #246,paid invoice,None,None,C001
1,869c1tu79,PARSONS NOSE (Putney) #246,paid invoice,None,None,C005
2,869bzfqnd,"SIGNE ,RUBER ROFF",paid invoice,None,None,C001
3,869bzfqnd,"SIGNE ,RUBER ROFF",paid invoice,None,None,C005
4,869bwemyd,EXEL (7A SCHUBERT ROAD SW15 2QT) #239,paid invoice,None,None,C002
...,...,...,...,...,...,...
58,8698nkayw,SHABHAN (LOFT PROJECT) 36 KERSLEY STREET,paid invoice,2025-04-09,2025-05-04,C044
59,8698m1cnu,KATE (ROOFING) 35 CLONCURRY STREET #178,paid invoice,2025-04-08,2025-04-09,C045
60,8698kr47j,LOTTIE DUNNE (PAINT) 8 DONARELLE STREET #176,paid invoice,2025-04-05,2025-04-05,C046
61,8698kr34y,PARSONS NOSE FULHAM (FRAM BOX) 5 HEATHMANS ROA...,paid invoice,2025-04-04,2025-04-04,C043


#### Material Fact

In [37]:
# --- Material Fact ---
material_fact = material_df.rename(columns={
        "expense_name": "expense_name",
        "expense_cost": "expense_cost",
        "quantity": "quantity"
    })
[["project_id", "expense_name", "expense_cost", "quantity"]]
material_fact

,project_id,project_name,expense_name,expense_cost,quantity
0,869c1tu79,PARSONS NOSE (Putney) #246,hps boiler,312.72,1.0
1,869c1tu79,PARSONS NOSE (Putney) #246,travis perkings,12.92,1.0
2,869bzfqnd,"SIGNE ,RUBER ROFF",raven ruber,480.95,1.0
3,869bzfqnd,"SIGNE ,RUBER ROFF",lords ply,31.44,1.0
4,869bzfqnd,"SIGNE ,RUBER ROFF",zyber proteck,15.39,1.0
...,...,...,...,...,...
278,8698kr34y,PARSONS NOSE FULHAM (FRAM BOX) 5 HEATHMANS ROA...,metal frame steel,84.19,1.0
279,8698kr34y,PARSONS NOSE FULHAM (FRAM BOX) 5 HEATHMANS ROA...,tools,9.88,1.0
280,8698kqvp0,SIOBHAN (LIGHT WELLS) 33 CLONCURRY STREET #174,materials,70.50,1.0
281,8698kqvp0,SIOBHAN (LIGHT WELLS) 33 CLONCURRY STREET #174,Floor Protection Materials,35.00,1.0


#### Project Fact

In [44]:
# --- Project Fact ---
# Aggregate materials to get total_material_cost
total_materials = (
        material_df.groupby("project_id")
        .apply(lambda x: (x["expense_cost"] * x["quantity"]).fillna(0).sum())
        .reset_index(name="total_material_cost")
    )
project_fact = cln_project_df[[
        "project_id", 
        "project_name",
        "project_value",
        "dardan_days_worked", 
        "musa_days_worked",
        "dori_days_worked", 
        "remzi_days_worked"
    ]].merge(total_materials, on="project_id", how="left")
project_fact

/var/folders/qv/dzzbgwwd3lx5227l6qt6cvmh0000gn/T/ipykernel_1725/3517528389.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x["expense_cost"] * x["quantity"]).fillna(0).sum())


,project_id,project_name,project_value,dardan_days_worked,musa_days_worked,dori_days_worked,remzi_days_worked,total_material_cost
0,869c1tu79,PARSONS NOSE (Putney) #246,662.72,0.5,None,0.5,None,325.640
1,869bzfqnd,"SIGNE ,RUBER ROFF",1860,1,1,1,1,527.780
2,869bwemyd,EXEL (7A SCHUBERT ROAD SW15 2QT) #239,350,None,None,None,None,25.780
3,869buyaht,ELAINE KLADER (36 School Road) #240,2450,2.5,2.5,2.5,None,367.800
4,869bu5vq4,parsons nose shops,1576,2,1.5,None,None,307.800
5,869brqjq1,TOM KING (103 NEW KINS ROAD) #236,707.39,1,1,None,None,257.390
6,869brqg9g,Victoria painting kichen,1200,2,2,None,None,231.710
7,869brqafr,MATTHEW ROYLE. CHEPSTOW CLOSE #235,2620,2,2,2,None,113.260
8,869bagzfk,IMOGEN - WALL (10A LOUVAINE ROAD ),1780,1,1,1,None,185.000
9,869b84yve,"KATY (3 Rushhill Mews, SW11 5NB)",None,None,None,None,None,136.150


## Export

In [45]:
# export.py
import pandas as pd

def export_to_excel(projects_df, materials_df, services_df, filename="/Users/madalinasamoila/besa_pipeline/data/raw/raw_data.xlsx"):
    with pd.ExcelWriter(filename, engine="xlsxwriter") as writer:
        projects_df.to_excel(writer, sheet_name="Projects", index=False)
        materials_df.to_excel(writer, sheet_name="Materials", index=False)
        services_df.to_excel(writer, sheet_name="Services", index=False)
        labor_df.to_excel(writer, sheet_name="Labor", index=False)
    print(f"✅ Data exported to {filename}")

In [47]:
import pandas as pd

def export_clean_tables_to_excel(
    client_dim,
    project_dim,
    project_fact,
    expense_fact,
    daily_labor_fact,
    filename="/Users/madalinasamoila/besa_pipeline/data/clean/clean_data.xlsx"
):
    tables = {
        "ClientDim": client_dim,
        "ProjectDim": project_dim,
        "ProjectFact": project_fact,
        "ExpenseFact": expense_fact,
        "DailyLaborFact": daily_labor_fact
    }

    with pd.ExcelWriter(filename, engine="xlsxwriter") as writer:
        for sheet_name, df in tables.items():
            if df is not None and not df.empty:
                df.to_excel(writer, sheet_name=sheet_name, index=False)
                worksheet = writer.sheets[sheet_name]

                # Auto-adjust column widths
                for i, col in enumerate(df.columns):
                    max_len = df[col].astype(str).map(len).max()
                    max_len = max(max_len, len(col)) + 2  # padding
                    worksheet.set_column(i, i, max_len)

    print(f"✅ Clean dimension/fact tables exported to {filename}")
